In [1]:
!pip install -q mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 143.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.0/221.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [4]:
import mlflow
import dagshub

dagshub.init(
    repo_owner="ansh777.tomar",
    repo_name="Youtube-Sentiment",
    mlflow=True
    )

#Set the environment
mlflow.set_experiment("Exp 4 - Handling Imbalanced Data")

Initialized MLflow to track repo "ansh777.tomar/Youtube-Sentiment"

Repository ansh777.tomar/Youtube-Sentiment initialized!

2026/08/20 12:40:16 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4 - Handling Imbalanced Data' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/15e5fcca5167451f83ff3506ab5d4c56', creation_time=1787229617048, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1787229617048, lifecycle_stage='active', name='Exp 4 - Handling Imbalanced Data', tags={}, trace_location=None, workspace='default'>

In [5]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [9]:
df = pd.read_csv('/content/dataset.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [ ]:
#Step 1 Function to run the experiment

def run_imbalanced_experiment(imbalance_method):
  ngram_range=(1,3)
  max_features=1000

  #Train Test Split
  X_train,X_test,y_train,y_test=train_test_split(df['clean_comment'],df['category'],test_size=0.2,random_state=42,stratify=df['category'])


  #Vectorization usinf TF-IDF
  vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
  X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
  X_test_vec = vectorizer.transform(X_test)  # Transform test data

  # Step 3: Handle class imbalance based on the selected method (only applied to the training set)
  if imbalance_method=='class_weights':
    #Use class weight in Random Forest
    class_weight='balanced'
  else:
    class_weight= None #Do not apply class weight if using resampling

    #Resampling Techniques(only apply to training set)
    if imbalance_method=='oversampling':
      smote=SMOTE(random_state=42)
      X_train_vec,y_train=smote.fit_resample(X_train_vec,y_train)
    elif imbalance_method=='adasyn':
      adasyn=ADASYN(random_state=42)
      X_train_vec,y_train=adasyn.fit_resample(X_train_vec,y_train)
    elif imbalance_method=='undersampling':
      rus=RandomUnderSampler(random_state=42)
      X_train_vec,y_train=rus.fit_resample(X_train_vec,y_train)
    elif imbalance_method=='smote_enn':
      smote_enn=SMOTEENN(random_state=42)
      X_train_vec,y_train=smote_enn.fit_resample(X_train_vec,y_train)

  # Step 5: Define and train a Random Forest model

  with mlflow.start_run() as run:

    # Set tags for the experiment and run
    mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
    mlflow.set_tag("experiment_type", "imbalance_handling")
    mlflow.set_tag("model_type", "RandomForestClassifier")

    # Add a description
    mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

    # Log vectorizer parameters
    mlflow.log_param("vectorizer_type", "TF-IDF")
    mlflow.log_param("ngram_range", ngram_range)
    mlflow.log_param("vectorizer_max_features", max_features)

    # Log Random Forest parameters
    n_estimators = 200
    max_depth = 15

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("imbalance_method", imbalance_method)

    # Initialize and train the model
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, class_weight=class_weight)
    model.fit(X_train_vec, y_train)

    # Step 6: Make predictions and log metrics
    y_pred = model.predict(X_test_vec)

    # Log accuracy
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Log classification report
    classification_rep = classification_report(y_test, y_pred, output_dict=True)
    for label, metrics in classification_rep.items():
      if isinstance(metrics, dict):
        for metric, value in metrics.items():
          mlflow.log_metric(f"{label}_{metric}", value)

    # Log confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
    confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
    plt.savefig(confusion_matrix_filename)
    mlflow.log_artifact(confusion_matrix_filename)
    plt.close()

    # Log the model
    mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}")

# Run Experiments for different imbalance methods
imbalance_methods=['class_weights','oversampling','adasyn','undersampling','smote_enn']

for method in imbalance_methods:
  run_imbalanced_experiment(method)

2026/08/20 14:11:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3/runs/408fea3eea2144289ffbd174e7d46fcc
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3


2026/08/20 14:13:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3/runs/8fe28fa14fc94cce9e0d93d82f23effc
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3


2026/08/20 14:14:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3/runs/b198d538913941f6ad56fc8e76f0d249
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3


2026/08/20 14:16:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_undersampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3/runs/241194f3b0614f4d8241885ee5b3a7d7
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3


2026/08/20 14:18:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_smote_enn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3/runs/1189b875b0584ae8892c8b9885fc640c
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/3
